# MovieLens-20M: shared loading and Level 1 EDA

This leader-owned notebook establishes the common data definitions, quality checks, derived tables, rating distribution, user activity baseline, genre baseline, and runtime feasibility note for the official MovieLens-20M dataset.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from load_data import load_movielens, summarize_frames
from preprocess import build_shared_tables, prepare_genome_scores, prepare_genome_tags
from metrics import data_quality_report, genre_summary, rating_distribution, user_activity_summary, activity_group_summary, runtime_feasibility
from plotting import plot_rating_distribution, plot_user_activity, plot_top_genres, plot_genre_rating_boxplot, save_figure

In [ ]:
data_dir = Path(os.environ.get('MOVIELENS_DATA_DIR', PROJECT_ROOT / 'data' / 'ml-20m'))
frames = load_movielens(data_dir)

ratings_raw = frames['ratings']
movies_raw = frames['movies']
tags_raw = frames.get('tags', pd.DataFrame(columns=['userId', 'movieId', 'tag', 'timestamp']))
print('Loaded data directory:', frames['_data_dir'])
display(summarize_frames(frames))

In [ ]:
tables = build_shared_tables(ratings_raw, movies_raw, tags_raw)
ratings = tables['ratings']
movies = tables['movies']
tags = tables['tags']
movie_stats = tables['movie_stats']
user_stats = tables['user_stats']
exploded_genres = tables['exploded_genres']
rating_genre_df = tables['rating_genre_df']
movie_tag_stats = tables['movie_tag_stats']

genome_scores = frames.get('genome-scores', pd.DataFrame())
genome_tags = frames.get('genome-tags', pd.DataFrame())
genome_scores = prepare_genome_scores(genome_scores) if not genome_scores.empty else None
genome_tags = prepare_genome_tags(genome_tags) if not genome_tags.empty else None

quality = data_quality_report(frames, ratings, movies, tags, genome_scores, genome_tags)
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'summary_tables'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
quality['file_summary'].to_csv(OUTPUT_DIR / 'file_summary.csv', index=False)
quality['key_integrity'].to_csv(OUTPUT_DIR / 'key_integrity.csv', index=False)
display(quality['file_summary'])
display(quality['key_integrity'])

## Dataset scale and date range

The assignment's date range is based on rating and tag timestamps. Release year is a separate movie metadata field and is never substituted for rating year.

In [ ]:
scale = pd.DataFrame({
    'metric': ['users', 'movies', 'ratings', 'tag applications', 'genome tags', 'rating start', 'rating end', 'tag start', 'tag end'],
    'value': [
        ratings['userId'].nunique(),
        movies['movieId'].nunique(),
        len(ratings),
        len(tags),
        genome_tags['tagId'].nunique() if genome_tags is not None else np.nan,
        ratings['rating_datetime'].min(),
        ratings['rating_datetime'].max(),
        tags['tag_datetime'].min() if not tags.empty else pd.NaT,
        tags['tag_datetime'].max() if not tags.empty else pd.NaT,
    ],
})
display(scale)
display(movies[['movieId', 'title', 'release_year', 'release_year_parse_ok']].head())

## Rating distribution

Ratings are treated as numeric observations; the code does not assume that only integer values occur.

In [ ]:
rating_counts, rating_stats = rating_distribution(ratings)
rating_counts.to_csv(OUTPUT_DIR / 'rating_counts.csv', index=False)
rating_stats.to_csv(OUTPUT_DIR / 'rating_summary.csv', index=False)
display(rating_stats)
display(rating_counts)
rating_fig = plot_rating_distribution(rating_counts)
save_figure(rating_fig, PROJECT_ROOT / 'figures' / 'rating_distribution.png')
plt.show()

## User activity baseline

User activity is the number of ratings given. Because the distribution is heavy-tailed, both linear and log-scaled views are exported. Activity groups are empirical percentile groups, not arbitrary fixed thresholds.

In [ ]:
user_activity_quantiles = user_activity_summary(user_stats)
activity_groups = activity_group_summary(user_stats)
user_activity_quantiles.to_csv(OUTPUT_DIR / 'user_activity_summary.csv', index=False)
activity_groups.to_csv(OUTPUT_DIR / 'activity_groups.csv', index=False)
display(user_activity_quantiles)
display(activity_groups)
activity_fig = plot_user_activity(user_stats)
save_figure(activity_fig, PROJECT_ROOT / 'figures' / 'ratings_per_user.png')
plt.show()
activity_log_fig = plot_user_activity(user_stats, log_scale=True)
save_figure(activity_log_fig, PROJECT_ROOT / 'figures' / 'ratings_per_user_log.png')
plt.show()

## Genre baseline

A multi-genre movie contributes to every genre it contains. Genre means are therefore rating-record means after exploding genres, while movie counts remain counts of unique movies.

In [ ]:
genre_stats = genre_summary(exploded_genres, rating_genre_df)
genre_stats.to_csv(OUTPUT_DIR / 'genre_summary.csv', index=False)
display(genre_stats.head(15))
genre_fig = plot_top_genres(genre_stats)
save_figure(genre_fig, PROJECT_ROOT / 'figures' / 'top_genres.png')
plt.show()
genre_box_fig = plot_genre_rating_boxplot(rating_genre_df, genre_stats)
save_figure(genre_box_fig, PROJECT_ROOT / 'figures' / 'top_genre_rating_boxplot.png')
plt.show()

## Runtime feasibility

Runtime is not inferred from title length. The result below records whether the loaded files contain runtime and whether external IMDb/TMDb identifiers are available for a documented join.

In [ ]:
runtime_result = runtime_feasibility(frames.get('links'), movies)
pd.DataFrame([runtime_result]).to_csv(OUTPUT_DIR / 'runtime_feasibility.csv', index=False)
display(pd.Series(runtime_result, name='value').to_frame())
print('Shared table shapes:')
display(pd.DataFrame({name: [len(frame)] for name, frame in tables.items() if isinstance(frame, pd.DataFrame)}).T.rename(columns={0: 'rows'}))

## Leader handoff notes

- The shared tables are movie_stats, user_stats, exploded_genres, rating_genre_df, and movie_tag_stats.
- Contributor notebooks should import these same helpers and keep rating year separate from release year.
- This notebook is configured for the official MovieLens-20M dataset before findings are copied into the final write-up.
- The final notebook applies the same definitions to Challenge Questions 1–5 and records the runtime limitation.